<a href="https://colab.research.google.com/github/namana1212/myprojects/blob/main/Kopia_notatnika_online_retail_dw_template_guided.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hurtownia danych – Online Retail (wersja z podpowiedziami)

## 0. Import bibliotek

👉 Wskazówka: użyj Pandas


In [ ]:
import pandas as pd
import numpy as np

## 1. Wczytanie danych

👉 Wskazówka:
- użyj `pd.read_csv`
- sprawdź `head()` i `info()`


In [5]:
# TODO
df = pd.read_csv("/content/sample_data/Online_Retail_sample.csv", encoding='ISO-8859-1')

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    3 non-null      int64  
 1   StockCode    3 non-null      object 
 2   Description  3 non-null      object 
 3   Quantity     3 non-null      int64  
 4   InvoiceDate  3 non-null      object 
 5   UnitPrice    3 non-null      float64
 6   CustomerID   3 non-null      int64  
 7   Country      3 non-null      object 
dtypes: float64(1), int64(3), object(4)
memory usage: 324.0+ bytes


## 2. Czyszczenie danych

👉 Wskazówki:
- usuń wiersze bez CustomerID → `dropna`
- usuń anulacje → InvoiceNo zaczyna się od 'C'
- usuń Quantity <= 0
- usuń UnitPrice <= 0

👉 Podpowiedź: filtruj dane krok po kroku


In [6]:
# usunięcie braków CustomerID
df = df.dropna(subset=['CustomerID'])

# usunięcie anulowanych transakcji
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# usunięcie rekordów z Quantity <= 0
df = df[df['Quantity'] > 0]

# usunięcie rekordów z UnitPrice <= 0
df = df[df['UnitPrice'] > 0]

# konwersja dat
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# usunięcie duplikatów
df = df.drop_duplicates()

# dodanie Revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536366,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:28:00,2.75,13047,United Kingdom,22.00


## 3. Wybór grain

👉 Pomyśl:
- czy jeden wiersz to produkt?
- czy faktura?

👉 Najczęstsza odpowiedź: linia faktury (InvoiceNo + StockCode)

**Opisz swoją decyzję (2-3 zdania)**



## 4. Schemat gwiazdy

👉 Minimum:
- FactSales
- DimCustomer
- DimProduct
- DimDate

👉 Pomyśl:
- jakie kolumny będą w każdej tabeli?


Zaprojektowałem schemat gwiazdy z jednej tabeli faktów "FactSales" oraz z 3 tabeli wybmiarów czyli:
- DimCustomer
- DimProduct
- DimDate

Tabela faktów bedzię przechowywać ilość oraz zysk:
- Quantity
- Revenue

Tabela DimCustomer bedzie zawieać informacje o klientach i kraju
Tabela DimProduct zawiera informacje o produktach
Tabela DimDate bedzie umożliwać analize w czasie.

## 5. Budowa wymiarów

👉 Wskazówki:
- DimCustomer → CustomerID, Country
- DimProduct → StockCode, Description
- DimDate → data z InvoiceDate

👉 Pamiętaj o kluczach sztucznych (ID)


In [7]:
# TODO: przykładowy schemat
# dim_customer = df[["CustomerID", "Country"]].drop_duplicates()
# dim_product = df[["StockCode", "Description"]].drop_duplicates()

#Zostawiam kod u góry jako podgląd

# tabela klientów
dim_customer = df[['CustomerID', 'Country']]
# usunięcie duplikatów
dim_customer = dim_customer.drop_duplicates()
# dodanie klucza sztucznego
dim_customer['CustomerKey'] = range(1, len(dim_customer)+1)
print(dim_customer.head())

# tabela produktów
dim_product = df[['StockCode', 'Description']]
# usunięcie duplikatów
dim_product = dim_product.drop_duplicates()
# klucz sztuczny
dim_product['ProductKey'] = range(1, len(dim_product)+1)
print(dim_product.head())


# tabela dat
dim_date = pd.DataFrame()
# unikalne daty
dim_date['Date'] = df['InvoiceDate'].dt.date.unique()
# klucz sztuczny
dim_date['DateKey'] = range(1, len(dim_date)+1)
# dodatkowe informacje o dacie
dim_date['Year'] = pd.to_datetime(dim_date['Date']).dt.year
dim_date['Month'] = pd.to_datetime(dim_date['Date']).dt.month
dim_date['Day'] = pd.to_datetime(dim_date['Date']).dt.day

print(dim_date.head())



   CustomerID         Country  CustomerKey
0       17850  United Kingdom            1
2       13047  United Kingdom            2
  StockCode                         Description  ProductKey
0    85123A  WHITE HANGING HEART T-LIGHT HOLDER           1
1     71053                 WHITE METAL LANTERN           2
2    84406B      CREAM CUPID HEARTS COAT HANGER           3
         Date  DateKey  Year  Month  Day
0  2010-12-01        1  2010     12    1


## 6. Budowa tabeli faktów

👉 Wskazówki:
- połącz dane z wymiarami
- dodaj miary:
  - Quantity
  - Revenue = Quantity * UnitPrice



In [8]:
# df["Revenue"] = df["Quantity"] * df["UnitPrice"]

#Tak jak wcześniej zostawiam kod u góry jako podgląd

# tworzymy kopię wyczyszczonych danych
fact_sales = df.copy()

# Revenue = Quantity * UnitPrice. Obliczamy przychód ze sprzedaży
fact_sales["Revenue"] = fact_sales["Quantity"] * fact_sales["UnitPrice"]


# łączymy tabelę faktów z wymiarem klientów dzięki temu do każdego klienta zostanie przypisany CustomerKey
fact_sales = fact_sales.merge(
    dim_customer[['CustomerID', 'CustomerKey']],
    on='CustomerID'
)


# łączymy tabelę faktów z wymiarem produktów do każdego produktu zostanie przypisany ProductKey
fact_sales = fact_sales.merge(
    dim_product[['StockCode', 'ProductKey']],
    on='StockCode'
)

# tworzymy pomocniczą kolumnę Date wyciągamy samą datę bez godziny
fact_sales['Date'] = fact_sales['InvoiceDate'].dt.date

# łączymy z tabelą dat aby uzyskać DateKey
fact_sales = fact_sales.merge(
    dim_date[['Date', 'DateKey']],
    on='Date'
)

# tabela faktów zawiera: klucze sztuczne oraz miary biznesowe
fact_sales = fact_sales[[
    'CustomerKey',
    'ProductKey',
    'DateKey',
    'Quantity',
    'Revenue'
]]


print(fact_sales.head())

   CustomerKey  ProductKey  DateKey  Quantity  Revenue
0            1           1        1         6    15.30
1            1           2        1         6    20.34
2            2           3        1         8    22.00


## 7. SCD – DimCustomer

👉 Wskazówka:
- wybierz SCD1 lub SCD2
- jeśli SCD2 → dodaj kolumny:
  - valid_from
  - valid_to
  - current_flag

👉 Pomyśl:
- czy chcesz historię?


Wybrałem SCD1 dla tabeli DimCustomer. W przypadku zmiany kraju to wartość jest nadpisywana a historia nie jest zapisywana. Jest tak samo w przypadku daty.

## 8. Wnioski

👉 Opisz:
- jakie decyzje podjąłeś
- czego się nauczyłeś
- jakie były trudności

Wybrałem jakie chce dokładnie tabele i co ma się w nich znajdować, również podjąłem decyzje dotyczącą SCD1 i SCD2.

Nauczyłem się mniej więcej jak działa hurtownia danych grain czy fact table.

Największą trudnością dla mnie było napisanie samego kodu, ponieważ mam z tym nie małe problemy aczkolwiek zrozumiałem same pojęcia całkiem łatwo.
